## Preprocessing

In [1]:
%pip install torch torchvision pandas pillow


  Using cached torch-2.10.0-cp313-cp313-win_amd64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp313-cp313-win_amd64.whl.metadata (5.4 kB)
Using cached torch-2.10.0-cp313-cp313-win_amd64.whl (113.8 MB)
Using cached torchvision-0.25.0-cp313-cp313-win_amd64.whl (4.3 MB)

   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 [torch]
   ---------------------------------------- 0/2 

In [7]:
import os
import cv2
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

In [8]:
def apply_clahe(pil_img):
    """
    Apply CLAHE to a PIL grayscale image and return a PIL image.
    Safely converts to 'L' mode and ensures uint8 dtype.
    """
    # ensure grayscale PIL image
    if pil_img.mode != "L":
        pil_img = pil_img.convert("L")

    img = np.array(pil_img)
    # handle float images in [0,1] or other dtypes
    if img.dtype != np.uint8:
        if img.max() <= 1.0:
            img = (img * 255).astype(np.uint8)
        else:
            img = img.astype(np.uint8)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img = clahe.apply(img)
    return Image.fromarray(img)


In [9]:
def load_grayscale(path):
    """
    Load image safely as grayscale
    """
    return Image.open(path).convert("L")


In [10]:
train_transform = T.Compose([
    T.Lambda(apply_clahe),
    T.Resize((224, 224)),
    T.RandomRotation(5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),  # scales to [0,1]
])


In [11]:
val_transform = T.Compose([
    T.Lambda(apply_clahe),
    T.Resize((224, 224)),
    T.ToTensor(),
])

In [12]:
class DentalCariesDataset(Dataset):
    def __init__(self, csv_file, image_dir, transform=None):
        self.df = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row["image_name"])

        image = load_grayscale(img_path)

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row["label"], dtype=torch.long)
        return image, label


In [13]:
# Dataset paths and quick sanity check
csv_path = "dentex_train_labels.csv"
img_dir = "C:\\Users\\nupur.sarkar\\Documents\\Dental_Xray_model\\quadrant-enumeration-disease\\xrays"  # or use an absolute path

# instantiate dataset and dataloader (make sure CELL INDEX: 7 with DentalCariesDataset is executed first)
# Note: Run the cell containing DentalCariesDataset class definition before running this cell
ds = DentalCariesDataset(csv_file=csv_path, image_dir=img_dir, transform=train_transform)
from torch.utils.data import DataLoader
dl = DataLoader(ds, batch_size=8, shuffle=True)

# load one sample to verify paths are correct
img, label = ds[0]
print('image tensor shape:', img.shape, 'label:', label)


image tensor shape: torch.Size([1, 224, 224]) label: tensor(1)


## Training


In [1]:
#Importing necessary libraries for model building and training
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import torchvision.models as models
import matplotlib.pyplot as plt
import cv2


In [ ]:
# Check for GPU availability and set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [27]:
from torch.utils.data import DataLoader

# Paths
CSV_PATH = "dentex_train_labels.csv"
IMAGE_DIR = "C:\\Users\\nupur.sarkar\\Documents\\Dental_Xray_model\\quadrant-enumeration-disease\\xrays"

# Dataset objects
train_dataset = DentalCariesDataset(
    csv_file=CSV_PATH,
    image_dir=IMAGE_DIR,
    transform=train_transform
)

val_dataset = DentalCariesDataset(
    csv_file=CSV_PATH,
    image_dir=IMAGE_DIR,
    transform=val_transform
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))


Train samples: 705
Val samples: 705


In [18]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("dentex_train_labels.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print("Train size:", len(train_df))
print("Val size:", len(val_df))

print("\nTrain label distribution:")
print(train_df["label"].value_counts())

print("\nVal label distribution:")
print(val_df["label"].value_counts())


Train size: 564
Val size: 141

Train label distribution:
label
1    520
0     44
Name: count, dtype: int64

Val label distribution:
label
1    130
0     11
Name: count, dtype: int64


In [19]:
train_df.to_csv("train_labels.csv", index=False)
val_df.to_csv("val_labels.csv", index=False)

In [20]:
import torch

class_counts = train_df["label"].value_counts().sort_index()
print(class_counts)

weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float)
weights = weights / weights.sum()  # normalize

print("Class weights:", weights)


label
0     44
1    520
Name: count, dtype: int64
Class weights: tensor([0.9220, 0.0780])


# Training the Model

In [21]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [22]:
import torch.nn as nn
import torchvision.models as models

def build_resnet18():
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Change first conv to 1 channel (grayscale)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )

    # Binary classification (2 logits)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

model = build_resnet18().to(device)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\nupur.sarkar/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:05<00:00, 9.03MB/s]


In [23]:
criterion = nn.CrossEntropyLoss(weight=weights.to(device))

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)


In [24]:
from tqdm import tqdm

def train_one_epoch(model, loader):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / len(loader)

In [25]:
import numpy as np
from sklearn.metrics import roc_auc_score, confusion_matrix

def validate(model, loader):
    model.eval()
    probs = []
    targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)

            # Probability of class 1 (caries)
            p = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()

            probs.extend(p)
            targets.extend(labels.numpy())

    auc = roc_auc_score(targets, probs)
    preds = (np.array(probs) > 0.5).astype(int)
    cm = confusion_matrix(targets, preds)

    return auc, cm


In [28]:
EPOCHS = 10

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader)
    val_auc, val_cm = validate(model, val_loader)

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val AUC: {val_auc:.4f}")
    print("Confusion Matrix:")
    print(val_cm)


100%|██████████| 89/89 [04:42<00:00,  3.18s/it]



Epoch 1/10
Train Loss: 0.6832
Val AUC: 0.8213
Confusion Matrix:
[[ 35  20]
 [ 79 571]]


100%|██████████| 89/89 [04:39<00:00,  3.14s/it]



Epoch 2/10
Train Loss: 0.5318
Val AUC: 0.8852
Confusion Matrix:
[[ 41  14]
 [107 543]]


100%|██████████| 89/89 [04:26<00:00,  3.00s/it]


...


Epoch 8/10
Train Loss: 0.2064
Val AUC: 0.9955
Confusion Matrix:
[[ 54   1]
 [ 33 617]]


100%|██████████| 89/89 [03:11<00:00,  2.15s/it]



Epoch 9/10
Train Loss: 0.1201
Val AUC: 1.0000
Confusion Matrix:
[[ 55   0]
 [  0 650]]


100%|██████████| 89/89 [03:09<00:00,  2.13s/it]



Epoch 10/10
Train Loss: 0.0487
Val AUC: 1.0000
Confusion Matrix:
[[ 55   0]
 [  0 650]]


In [29]:
print(train_dataset.df.shape)
print(val_dataset.df.shape)

print(train_dataset.df.head())
print(val_dataset.df.head())


(705, 2)
(705, 2)
      image_name  label
0  train_673.png      1
1  train_283.png      1
2  train_435.png      0
3   train_95.png      1
4  train_475.png      1
      image_name  label
0  train_673.png      1
1  train_283.png      1
2  train_435.png      0
3   train_95.png      1
4  train_475.png      1
